# XRSlab quality-gated analysis

This notebook is the interactive workbench. All computation is delegated to `xrslab.workflow`; the notebook only configures, reviews, approves and exports.

In [ ]:
from xrsabre.paths import load_workspace
from xrsabre.notebook import missing_scan_ids, run_qc_review
from xrslab.roi_editor import RoiEditor
from xrslab.workflow import (
    AnalysisConfig, QcApproval, build_qc_report, export_analysis,
    finalize_analysis, prepare_analysis,
)

## 1. Configure

This is the only analysis-parameter cell. Paths are resolved at runtime and are not stored in the portable configuration.

In [ ]:
# Analysis configuration
config = AnalysisConfig(
    element='Ho',
    elastic_scan_ids=(57,),
    xrs_scan_ids=(59,),
    analysis_name='XRS_analysis',
    modules=('VB', 'HB', 'HL', 'VD', 'VU'),
    q_range=(0.0, 10.0),
    auto_adjust_rois=True,
    filter_value=0.15,
    elastic_center_range_kev=(9.67, 9.69),
    max_fwhm_ev=2.0,
    min_r_squared=0.8,
    energy_step_ev=0.2,
)
workspace = load_workspace()
print(config)
print('raw:', workspace.raw / config.element)
print('processed:', workspace.processed)
missing_scans = missing_scan_ids(config, workspace)
data_ready = not missing_scans
if data_ready:
    print('All configured raw scans are available.')
else:
    print('Raw data is not ready; the analysis cells will be skipped.')
    print('Add these scan directories under', workspace.raw / config.element, ':', missing_scans)

## 2a. Optional interactive ROI editing

Run the next cell when you want to inspect or redraw ROIs before the full pipeline. The editor loads only elastic scan 57. Drag a selected rectangle or its handles to move and resize it; use the controls to add, delete, undo, redo, reset and save. Saving creates versioned files and returns a configuration with further automatic ROI adjustment disabled.

In [ ]:
# Optional: initialise the interactive backend and load only the elastic scan.
roi_editor = None
if data_ready:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('matplotlib', 'widget')
    roi_editor = RoiEditor.from_config(config, workspace)
    roi_editor.display()
else:
    print('ROI editor skipped until the configured raw scans are copied into the workspace.')

In [ ]:
# Run after clicking '另存版本' in the editor. Skip this cell if no edit was saved.
if data_ready and roi_editor is not None and roi_editor.last_result is not None:
    config = roi_editor.last_result.config
    print('Using manual ROI files:', roi_editor.last_result.filenames)
    print('Automatic ROI adjustment:', config.auto_adjust_rois)
elif data_ready:
    print('No manual ROI version has been saved; configuration is unchanged.')
else:
    print('No manual ROI version can be saved until raw data is available.')

## 2. Prepare, calibrate and review QC

This stage validates NeXus inputs, corrects and normalises I0, calibrates ROIs, fits elastic peaks, integrates XRS spectra and builds the QC report. It does not export formal results.

In [ ]:
if data_ready:
    prepared, qc = run_qc_review(config, workspace)
else:
    prepared = None
    qc = None
    print('QC review skipped. Copy the required raw NXS scans, then rerun this cell.')

## 3. Explicit QC approval

Review the diagnostics above. Add scan IDs or canonical ROI IDs such as `lambda:VU-E1` to the exclusions. Change `approved` to `True` only after review.

In [ ]:
approval = QcApproval(
    approved=False,
    excluded_scans=(),
    excluded_rois=(),
    note='Reviewed ROI, I0, fit and coverage diagnostics',
)
if data_ready:
    assert prepared is not None
    result = finalize_analysis(prepared, approval)
    print('approved:', result.approved)
    print('selected ROI:', len(result.selected_roi_ids))
    print('used XRS scans:', result.used_scan_ids)
else:
    result = None
    print('Approval is unavailable until raw data is available.')

## 4. Export immutable run

Formal export is refused unless the approval above is explicit. Each successful export creates a new provenance-rich run directory.

In [ ]:
if data_ready and approval.approved and result is not None and qc is not None:
    output_path = export_analysis(
        result, qc, config, workspace,
    )
    print('saved to:', output_path)
else:
    print('QC has not been approved; no formal output was written.')